In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 1. Data understanding

In [ ]:
df = pd.read_parquet("data/df_train_full_raw.parquet")
print(df.shape)

In [ ]:
train_base = pd.read_parquet(
    "data/train/train_base.parquet", columns=["case_id", "date_decision"])
dates = df[["case_id"]].merge(
    train_base, on="case_id", how="left")["date_decision"]
print("Exact date range:", dates.min(), "to", dates.max())
print("Year range:", df["year_decision"].min(),
      "to", df["year_decision"].max())
print("WEEK_NUM range:", df["WEEK_NUM"].min(
), "to", df["WEEK_NUM"].max())
print("Number of distinct weeks:", df["WEEK_NUM"].nunique())

### 1.1 Aantal non defaults vs aantal defaults

In [ ]:
target_counts = df["target"].value_counts().sort_index()
colors = {0: "blue", 1: "red"}

fig, ax = plt.subplots(figsize=(5, 5))
bars = ax.bar(
    target_counts.index,
    target_counts.values,
    color=[colors[t] for t in target_counts.index],
    width=0.6,
)

max_count = target_counts.values.max()
for bar, count in zip(bars, target_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_count * 0.015,
        f"{count:,}",
        va="bottom",
        ha="center",
        fontsize=10,
    )

ax.set_xticks(target_counts.index)
ax.set_xticklabels(["0 (non-default)", "1 (default)"])
ax.set_ylabel("Aantal cases")
ax.set_ylim(0, max_count * 1.15)
ax.set_title("Aantal defaults vs. non-defaults")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

### 1.2 Missing value's

In [ ]:
id_cols = ["case_id", "WEEK_NUM", "target"]
missing_pct = (df.drop(columns=id_cols).isna().mean()
               * 100).sort_values(ascending=False)

# Fig
fig, ax_hist = plt.subplots(figsize=(10, 5))

counts, bin_edges, patches = ax_hist.hist(missing_pct, bins=20, color="blue")

# Label each bin with its feature count (skip empty bins to avoid clutter)
for count, left, right in zip(counts, bin_edges[:-1], bin_edges[1:]):
    if count > 0:
        ax_hist.text(
            (left + right) / 2,
            count + counts.max() * 0.015,
            f"{int(count)}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

ax_hist.set_xlabel("Missing (%)")
ax_hist.set_ylabel("Aantal features")
ax_hist.set_title(f"Verdeling van missing % (n=465 features)")
ax_hist.set_ylim(0, counts.max() * 1.15)
ax_hist.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

# List
missing_table = missing_pct.reset_index()
missing_table.columns = ["feature", "missing_%"]
missing_table

### 1.3 Default rate per maand + aantal leningen per maand + corona periode aangeduid

In [ ]:
# Build a period column (year-month) and compute monthly default rate
df["period"] = pd.to_datetime(
    df["year_decision"].astype(
        str) + "-" + df["month_decision"].astype(str).str.zfill(2)
)

monthly = (
    df.groupby("period")["target"]
    .agg(default_rate="mean", n="count")
    .reset_index()
    .sort_values("period")
)

# COVID window: March 2020 – June 2021 (capped at last data point)
covid_start = pd.Timestamp("2020-03-01")
covid_end = min(pd.Timestamp("2021-06-30"), monthly["period"].max())

fig, ax1 = plt.subplots(figsize=(12, 4))

# Bar chart: number of loans (background, secondary axis)
ax2 = ax1.twinx()
ax2.bar(monthly["period"], monthly["n"], width=20,
        color="steelblue", alpha=0.2, label="Aantal leningen")
ax2.set_ylabel("Aantal leningen")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{int(y):,}"))

# Line: default rate (foreground, primary axis)
ax1.plot(monthly["period"], monthly["default_rate"] * 100,
         color="red", linewidth=2, zorder=3, label="Default rate")
ax1.set_xlabel("Datum")
ax1.set_ylabel("Default rate (%)")
ax1.set_title("Maandelijkse default rate en aantal leningen over tijd")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.1f}%"))
ax1.set_zorder(ax2.get_zorder() + 1)
ax1.patch.set_visible(False)

# COVID shading
ax1.axvspan(covid_start, covid_end, color="red", alpha=0.12,
            label="COVID-19 periode\n(mrt 2020 – jun 2021)")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

fig.tight_layout()
plt.show()